# DocMatcher GNN Remote GPU Workflow

This notebook is a direct-run workflow for a remote JupyterLab GPU machine.

What it does:
- installs dependencies in the current notebook kernel
- runs a baseline DocMatcher sanity check
- optionally fine-tunes the GNN matcher
- runs full DocMatcher inference with a custom GNN LightGlue checkpoint
- evaluates the output run

Notes:
- Baseline `docmatcher@inv3d` uses the released LightGlue checkpoint.
- The GNN matcher is opt-in and should be fine-tuned before claiming results.
- Full GNN training requires the Inv3D train/val/test data under `input/inv3d/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
GPU = 0

# Quick toggles
INSTALL_TORCH = False
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu118"
RUN_BASELINE_SANITY = True
RUN_GNN_TRAINING = False
RUN_GNN_INFERENCE = True
RUN_EVAL = True

# GNN settings
GRAPH_K_NEIGHBORS = 5
GRAPH_SPARSE_ATTENTION = False
EXPERIMENT_NAME = "inv3d_former2_glue1_gnn"

# Inference targets
BASELINE_DATASET = "example"      # change to "inv3d_real" later
GNN_DATASET = "example"           # change to "inv3d_real" after training

print(f"Repo root: {REPO_ROOT}")
print(f"Python:    {sys.executable}")

In [ ]:
def run_cmd(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)

run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
if INSTALL_TORCH:
    run_cmd([
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "torchvision",
        "--index-url",
        TORCH_INDEX_URL,
    ])
run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

## Baseline Sanity Check

This verifies that the released DocMatcher pipeline runs in the current environment.

In [ ]:
if RUN_BASELINE_SANITY:
    run_cmd([
        sys.executable,
        "inference.py",
        "--model",
        "docmatcher@inv3d",
        "--dataset",
        BASELINE_DATASET,
        "--gpu",
        str(GPU),
    ])
else:
    print("Skipping baseline sanity run.")

## Optional: Fine-Tune the GNN Matcher

Enable `RUN_GNN_TRAINING = True` only if the full `input/inv3d/train`, `val`, and `test` data are available.

In [ ]:
if RUN_GNN_TRAINING:
    cmd = [
        sys.executable,
        "train.py",
        "--model-part",
        "lightglue",
        "--gpu",
        str(GPU),
        "--enable-gnn",
        "--graph-k-neighbors",
        str(GRAPH_K_NEIGHBORS),
        "--experiment-name",
        EXPERIMENT_NAME,
    ]
    if GRAPH_SPARSE_ATTENTION:
        cmd.append("--graph-sparse-attention")
    run_cmd(cmd)
else:
    print("Skipping GNN training.")

## Find the GNN Checkpoint

If you trained the GNN matcher in this notebook, this will usually find `last.ckpt`.
If you already trained it earlier, adjust `EXPERIMENT_NAME` above.

In [ ]:
checkpoint_root = REPO_ROOT / "models" / "training" / "lightglue" / EXPERIMENT_NAME
all_checkpoints = sorted(checkpoint_root.rglob("*.ckpt"))
print("Checkpoint root:", checkpoint_root)
for ckpt in all_checkpoints[-10:]:
    print(ckpt)

last_candidates = sorted(checkpoint_root.rglob("last.ckpt"))
GNN_CHECKPOINT = str(last_candidates[-1]) if last_candidates else None
print("Selected GNN checkpoint:", GNN_CHECKPOINT)

## Run Full DocMatcher with the GNN Matcher

This uses the trained LightGlue GNN checkpoint in the line-matching stage of the full pipeline.

In [ ]:
if RUN_GNN_INFERENCE:
    if not GNN_CHECKPOINT:
        raise FileNotFoundError(
            "No GNN checkpoint found. Train first or set EXPERIMENT_NAME to an existing run."
        )
    cmd = [
        sys.executable,
        "inference.py",
        "--model",
        "docmatcher@inv3d",
        "--dataset",
        GNN_DATASET,
        "--gpu",
        str(GPU),
        "--enable-gnn-line-matcher",
        "--graph-k-neighbors",
        str(GRAPH_K_NEIGHBORS),
        "--lightglue-checkpoint",
        GNN_CHECKPOINT,
    ]
    if GRAPH_SPARSE_ATTENTION:
        cmd.append("--graph-sparse-attention")
    run_cmd(cmd)
else:
    print("Skipping GNN inference.")

## Evaluate the Output Run

In [ ]:
if RUN_EVAL:
    run_name = f"{GNN_DATASET}-docmatcher@inv3d"
    run_cmd([sys.executable, "eval.py", "--run", run_name])
else:
    print("Skipping evaluation.")

## Suggested Sequence

1. Run the install cell.
2. Run the baseline sanity check on `example`.
3. If baseline works and the full dataset is mounted, set `RUN_GNN_TRAINING = True` and train.
4. Confirm `GNN_CHECKPOINT` is found.
5. Run GNN inference on `example`.
6. Switch `GNN_DATASET` to `inv3d_real` for the real run.
7. Compare the baseline and GNN metrics.